# Exploratory Data Analysis: eBird Abundance Rasters

Load weekly abundance GeoTIFFs, compute movement metrics, and visualize patterns for migration season detection.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path("..").resolve()))

import numpy as np
import matplotlib.pyplot as plt
from src.raster_processing import load_weekly_stack, find_species_data, get_season_dates
from src.feature_extraction import compute_global_features

In [ ]:
# Load weekly abundance (52 weeks)
data_dir = Path("../data/raw")
stack, meta = load_weekly_stack(data_dir, species="yebsap-example", resolution="27km")
print(f"Shape: {stack.shape} (weeks, height, width)")
print(f"Dates: {meta['dates'][:5]} ... {meta['dates'][-3:]}")

In [ ]:
# Compute movement features
features = compute_global_features(stack)
for k, v in features.items():
    print(f"{k}: shape {v.shape}")

In [ ]:
# Load season labels from config
species_dir = find_species_data(data_dir, "yebsap-example")
seasons = get_season_dates(species_dir)
for s in seasons:
    print(f"{s['season']}: {s['start_date']} to {s['end_date']}")

In [ ]:
# Plot movement metrics over time
fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)
weeks = np.arange(len(meta["dates"]))

axes[0].plot(weeks, features["centroid_row"], label="centroid row")
axes[0].plot(weeks, features["centroid_col"], label="centroid col")
axes[0].set_ylabel("Centroid")
axes[0].legend()

axes[1].plot(weeks[1:], features["centroid_displacement"], label="displacement")
axes[1].plot(weeks[1:], features["change_magnitude"] / 1e6, label="change (M)")
axes[1].set_ylabel("Movement")
axes[1].legend()

axes[2].plot(weeks, features["spatial_variance"], label="variance")
axes[2].plot(weeks, features["spatial_entropy"], label="entropy")
axes[2].set_ylabel("Spread")
axes[2].set_xlabel("Week")
axes[2].legend()

plt.suptitle("Global Movement Metrics (Yellow-bellied Sapsucker)")
plt.tight_layout()
plt.show()

In [ ]:
# Sample weeks: abundance maps
fig, axes = plt.subplots(2, 4, figsize=(14, 6))
sample_weeks = [0, 12, 20, 26, 32, 38, 44, 50]  # spread across year
for ax, w in zip(axes.ravel(), sample_weeks):
    im = ax.imshow(stack[w], cmap="YlOrRd", vmin=0, vmax=np.nanpercentile(stack, 95))
    ax.set_title(meta["dates"][w])
    ax.axis("off")
plt.suptitle("Weekly Abundance (sample weeks)")
plt.tight_layout()
plt.show()